In [ ]:
from THBSplines.src.hierarchical_space import HierarchicalSpace
from THBSplines.src.cartesian_mesh import CartesianMesh
import numpy as np
import scipy.sparse as sp
import dolfinx
from mpi4py import MPI
import basix.ufl
import pyvista

import cffi
import numba
import numba.core.typing.cffi_utils as cffi_support
from dolfinx.jit import ffcx_jit
from dolfinx import default_real_type, default_scalar_type, geometry
rtype = default_real_type
dtype = default_scalar_type
import ufl
from ffcx.codegeneration.utils import empty_void_pointer
from ffcx.codegeneration.utils import numba_ufcx_kernel_signature as ufcx_signature

import numpy.typing as npt


def refine(knots: npt.ArrayLike, p: int, n_times: int=1)->npt.NDArray:
    """Given `knots`, returns its dyadic refinement with multiplicity `p+1`
    at the extremities."""
    knots: npt.NDArray[np.float_] = np.asarray(knots)
    mult_left: int = np.searchsorted(knots, knots[0], side='right')
    mult_right: int = len(knots) - np.searchsorted(knots, knots[-1], side='left')
    pad_left: int = max(0, p + 1 - mult_left)
    pad_right: int = max(0, p + 1 - mult_right)
    if pad_left > 0 or pad_right > 0:
        knots = np.concatenate((
            np.full(pad_left, knots[0], dtype=knots.dtype),
            knots,
            np.full(pad_right, knots[-1], dtype=knots.dtype)
        ))
    if n_times == 0:
        return knots
    
    # Find indices where the knot value changes
    jump_idx: npt.NDArray[np.int_] = np.where(knots[1:] > knots[:-1])[0]
    left_vals = knots[jump_idx]
    right_vals =knots[jump_idx + 1]
    num_new_points = (1<<n_times)-1
    fractions = np.linspace(0.,1.,num_new_points+2)[1:-1]
    new_points = left_vals[:, None] + (right_vals - left_vals)[:, None] * fractions[None, :]
    new_points = new_points.ravel()
    insert_positions = np.repeat(jump_idx + 1, num_new_points)
    
    return np.insert(knots, insert_positions, new_points)
        

In [ ]:
n_refinements = 1
p0 = 2
knots1 = np.array([0,0,0, 0.5,0.5,1, 1, 1], dtype=np.float64)
#knots1 = np.array([-1, -1, -1, 0,0,1,1,1], dtype=np.float64)
knots1 = refine(knots1, p=p0, n_times=n_refinements)
log_initial_mesh_size = np.log2(np.max(np.diff(knots1)))
knots2 = np.array([0,0,0,0.5,1,1,1], dtype=np.float64)
#knots2 = np.array([-1, -1, -1, 0,0,1,1,1], dtype=np.float64)
knots2 = refine(knots2, p0, n_times=n_refinements-1)
err_cells = {}
hs = HierarchicalSpace(knots=[knots1, knots2], degrees=[p0])
area_to_exclude = np.array([[0,1.], [-1., 0.]], dtype=np.float64)
area_to_refine = np.array([[0.25, 0.75], [0., 0.5]])


In [ ]:
err_cells

In [ ]:
for level, cells in err_cells.items():
    #print(cells)
    hs.refine(cells, level, refine_neighbours=True)


In [ ]:
# forbidden_cells = {}
# for level in range(hs.nlevels):
#     my_inactive_cells = hs.hmesh.get_indices_in_rectangle(area_to_exclude, level)
#     forbidden_cells[level] = my_inactive_cells
#     active_cells_l = hs.hmesh.aelem_level[level]
#     #print(f"level={level}, inactive_cells = {my_inactive_cells}")
#     #multi_coords = np.unravel_index(active_cells_l, hs.hmesh.meshes_shape[l])
#     active_cells_l = np.setdiff1d(active_cells_l, my_inactive_cells)
#     hs.hmesh.aelem_level[level] = active_cells_l
#     hs.hmesh.delem_level[level] = my_inactive_cells
#     #print(f"level=level, forbidden_cells=")
# hs._update_active_functions(no_overlap_on=forbidden_cells)
hs.hmesh.plot_cells()

In [ ]:
total_active_cells = sum(len(hs.hmesh.aelem_level[l]) for l in range(hs.nlevels))

all_cells = np.empty((4*total_active_cells, 2), dtype=np.float64) # will have coarser cells on top and finer on bottom
thb_operators: dict[tuple[int, int], npt.NDArray[np.float_]] = {}
N_max = 0 # maximum amount of dofs in a cell
current_idx = 0
for l in range(hs.nlevels):
    active_cells_l = hs.hmesh.aelem_level[l]
    
    if len(active_cells_l)==0:
        continue
   
    thb_operators_list = hs.local_multi_level_extraction_operator2(active_cells_l, l, l)
    thb_operators.update({(l, cell): op for cell, op in zip(active_cells_l, thb_operators_list)})
    
    if thb_operators_list:
        level_max = max(op.shape[0] for op in thb_operators_list)
        N_max = max(N_max, level_max)

    mesh = CartesianMesh(hs.hmesh.one_d_indices[l], len(hs.hmesh.one_d_indices[l]))
    my_cells_l = mesh.cells[active_cells_l]

    n_cells = len(my_cells_l)
    x_coords = my_cells_l[:, 0, :]
    y_coords = my_cells_l[:, 1, :] 

    start, end = current_idx, current_idx+(4*n_cells)

    view = all_cells[start:end]
    #points = np.zeros((4 * len(my_cells_l), 2), dtype=np.float64)
    view[::4] = np.column_stack((x_coords[:, 0], y_coords[:, 0]))  # Bottom-left
    view[1::4] = np.column_stack((x_coords[:, 1], y_coords[:, 0]))  # Bottom-right
    view[2::4] = np.column_stack((x_coords[:, 0], y_coords[:, 1]))  # Top-left
    view[3::4] = np.column_stack((x_coords[:, 1], y_coords[:, 1]))  # Top-right

    current_idx=end
pass
del mesh # free this big object

#all_cells = np.array(all_cells).reshape(-1, 2)

#coordinates = np.arange(len(all_cells), dtype=np.int32).reshape(-1, 4)
#coordinate_element = basix.ufl.element("Q", "quadrilateral", 1, shape=(2,))
#disconnected_mesh = dolfinx.mesh.create_mesh(MPI.COMM_WORLD, cells=coordinates, e=coordinate_element, x=all_cells)

all_cells = np.array(all_cells).reshape(-1, 2)

def map_uv_to_xy_small(uv_points, nodes_per_cell=4):
    original_shape = uv_points.shape
    uv_flat = uv_points.reshape(-1, 2)
    
    u = uv_flat[:, 0]
    v = uv_flat[:, 1]
    xy_flat = np.zeros_like(uv_flat)

    # ---------------------------------------------------------
    # MACRO PATCH 1: Left Half (u < 0.5)
    # ---------------------------------------------------------
    left_mask = u < 0.5
    u_L = u[left_mask]
    v_L = v[left_mask]
    # P(U,V) = (1-U)(1-V)P00 + U(1-V)P10 + (1-U)VP01 + UVP11
    # for this specific problem, 
    # P00=(0,-1), 
    # P10 =(0,0)
    # P01 = (-1, -1), 
    # P11 = (-1, 1)
    
    xy_flat[left_mask, 0] = -v_L
    # substituting U=2u, V=v
    xy_flat[left_mask, 1] = 2 * u_L * v_L + 2 * u_L - 1

    # ---------------------------------------------------------
    # MACRO PATCH 2: Right Half (u >= 0.5)
    # ---------------------------------------------------------
    right_mask = u >= 0.5
    u_R = u[right_mask]
    v_R = v[right_mask]
    
    xy_flat[right_mask, 0] = 2 * u_R * v_R + 2 * u_R - 2 * v_R - 1
    xy_flat[right_mask, 1] = v_R
    
    return xy_flat.reshape(original_shape)


def map_uv_to_xy_turn(uv_points, nodes_per_cell=4):
    """      C00, C10, C01, C11 = [-0.5, -1], [0, -1], [-0.5, -0.25], [0, -0.5]
                u_loc, v_loc = u * 4.0, v * 2.0
                
                
        elif u_c < 0.5:
            if v_c >= 0.5:
                # Cell 1 -> P2
                C00, C10, C01, C11 = [-1, 0], [-0.5, -0.25], [-1, 1], [-0.5, 0.5]
                u_loc, v_loc = (u-0.25) * 4.0, (v - 0.5) * 2.0
            else:
                # Cell 3 -> P3 
                C00, C10, C01, C11 = [-0.5, -0.25], [0, -0.5], [-0.5, 0.5], [0, 0]
                u_loc, v_loc = (u - 0.25) * 4.0, v * 2.0
                
        elif u_c < 0.75:
         Maps parametric [0,1]^2 to a 4-cell L-shape without polar singularities."""
    # Ensure points are structured as complete cells
    assert len(uv_points) % nodes_per_cell == 0, "uv_points must be grouped by cells."
    
    # Reshape points to (N_cells, 4, 2)
    uv_cells = uv_points.reshape(-1, nodes_per_cell, 2)
    xy_cells = np.zeros_like(uv_cells)
    
    def bilinear(u_loc, v_loc, C00, C10, C01, C11):
        """Standard isoparametric Q1 interpolation"""
        return ((1-u_loc)*(1-v_loc)*C00 + 
                u_loc*(1-v_loc)*C10 + 
                (1-u_loc)*v_loc*C01 + 
                u_loc*v_loc*C11)
    
    # Map cell by cell using the cell center to uniquely identify the block
    for i in range(len(uv_cells)):
        u = uv_cells[i, :, 0]
        v = uv_cells[i, :, 1]
        
        # Cell centers bypass boundary ambiguities
        u_c = np.mean(u)
        v_c = np.mean(v)
        
        if u_c < 0.25:
            if v_c >= 0.5:
                # Cell 0 -> P0 (Bottom-Left)
                C00, C10, C01, C11 = [-0.5, -1], [-0.5, -0.25], [-1, -1], [-1, 0]
                u_loc, v_loc = u * 4.0, (v-0.5) * 2.0
            else:
                # Cell 2 -> P1 
                C00, C10, C01, C11 = [0, -1], [0, -0.5], [-0.5, -1], [-0.5, -0.25]
                u_loc, v_loc = u * 4.0, v * 2.0
                
                
        elif u_c < 0.5:
            if v_c >= 0.5:
                # Cell 1 -> P2
                C00, C10, C01, C11 = [-0.5, -0.25], [-0.5, 0.5], [-1, 0], [-1, 1]
                u_loc, v_loc = (u-0.25) * 4.0, (v - 0.5) * 2.0
            else:
                # Cell 3 -> P3 
                C00, C10, C01, C11 = [0, -0.5], [0, 0], [-0.5, -0.25], [-0.5, 0.5]
                u_loc, v_loc = (u - 0.25) * 4.0, v * 2.0
                
        elif u_c < 0.75:
            if v_c < 0.5:
                # Cell 4 -> P4 (Bottom-Mid-Right) - Connects to P3's top edge!
                C00, C10, C01, C11 = [0, 0], [0.5, 0], [-0.5, 0.5], [0.25, 0.5]
                u_loc, v_loc = (u - 0.5) * 4.0, v * 2.0
            else:
                # Cell 5 -> P5 (Top-Mid-Right)
                C00, C10, C01, C11 = [-0.5, 0.5], [0.25, 0.5], [-1, 1], [0, 1]
                u_loc, v_loc = (u - 0.5) * 4.0, (v - 0.5) * 2.0
                
        else:
            if v_c < 0.5:
                # Cell 6 -> P6 (Bottom-Right)
                C00, C10, C01, C11 = [0.5, 0], [1, 0], [0.25, 0.5], [1, 0.5]
                u_loc, v_loc = (u - 0.75) * 4.0, v * 2.0
            else:
                # Cell 7 -> P7 (Top-Right)
                C00, C10, C01, C11 = [0.25, 0.5], [1, 0.5], [0, 1], [1, 1]
                u_loc, v_loc = (u - 0.75) * 4.0, (v - 0.5) * 2.0
                
        C00, C10 = np.array(C00), np.array(C10)
        C01, C11 = np.array(C01), np.array(C11)
        
        xy = bilinear(u_loc[:, None], v_loc[:, None], C00, C10, C01, C11)
        xy_cells[i] = xy
        
    return xy_cells.reshape(-1, 2)


all_cells_physical = map_uv_to_xy_small(all_cells)

physical_cells_midpoints = np.mean(all_cells_physical.reshape(-1, 4, 2), axis=1)

coordinates = np.arange(len(all_cells_physical), dtype=np.int32).reshape(-1, 4)
coordinate_element = basix.ufl.element("Q", "quadrilateral", 1, shape=(2,))
disconnected_mesh = dolfinx.mesh.create_mesh(MPI.COMM_WORLD, cells=coordinates, e=coordinate_element, x=all_cells_physical)

In [ ]:
# topology, cell_types, geometry = dolfinx.plot.vtk_mesh(disconnected_mesh)
# grid = pyvista.UnstructuredGrid(topology, cell_types, geometry)

# plotter = pyvista.Plotter()
# plotter.add_mesh(grid.shrink(.95), show_edges=False, color="#03bb85")
# plotter.view_xy()
# plotter.show(jupyter_backend="static")
# print(f"Number of points in PyVista grid: {grid.n_points}")

In [ ]:
for l in range(hs.nlevels):
    active_cells_l = hs.hmesh.aelem_level[l]
    for cell in active_cells_l:
        current_op = thb_operators[l, cell]
        shape = current_op.shape
        #print(f"shape = {shape}")
        axis_sum = current_op.sum(axis=0)
        if not np.allclose(axis_sum, np.ones((len(axis_sum)))):
            print(f"level= {l}, cell={cell}, operator = \n {axis_sum}\n")

In [ ]:
def outer_boundary(x):
    on_left = np.isclose(x[0], -1.)
    on_bottom = np.isclose(x[1], -1.)
    on_right = np.isclose(x[0], 1.) & (x[1]>=-1e-10)
    on_top = np.isclose(x[1], 1.) & (x[0]<=1.)
    return on_left|on_bottom|on_right|on_top

legendre_elt = basix.ufl.element(
    "DG",
    "quadrilateral",
    degree=p0,
    lagrange_variant=basix.LagrangeVariant.legendre
)
V = dolfinx.fem.functionspace(disconnected_mesh, legendre_elt)
print(f"Number of degrees of freedom: {V.dofmap.index_map.size_global}")
facet_dim = disconnected_mesh.topology.dim-1
boundary_facets = dolfinx.mesh.locate_entities_boundary(disconnected_mesh, facet_dim, outer_boundary)
custom_metadata = {"quadrature_degree": 12}
#ds = ufl.Measure("ds", domain=disconnected_mesh, subdomain_data=([1, boundary_facets]), metadata=custom_metadata)
dx_custom = ufl.Measure("dx", domain=disconnected_mesh, metadata=custom_metadata)


u,v = ufl.TrialFunction(V), ufl.TestFunction(V) 
f = dolfinx.fem.Function(V)
#f.interpolate(lambda x: x[0]*x[1]+0.9*x[0]**2-.7)
f.interpolate(lambda x: 1./(1.*np.exp((x[0]+.0625)**2 + (x[1]-.0625)**2)))
a0 = ufl.inner(u,v)*dx_custom
f0 = ufl.inner(f,v)*dx_custom
f_square_integral = dolfinx.fem.assemble_scalar(dolfinx.fem.form(ufl.inner(f, f)*dx_custom))
f_sq_integral = np.sqrt(disconnected_mesh.comm.allreduce(f_square_integral, op=MPI.SUM))


msh = disconnected_mesh
ufcxa0, _, _ = ffcx_jit(msh.comm, a0, form_compiler_options={"scalar_type": dtype})  # type: ignore
kernela0 = getattr(ufcxa0.form_integrals[0], f"tabulate_tensor_{np.dtype(dtype).name}")  # type: ignore
ufcxf0, _, _ = ffcx_jit(msh.comm, f0, form_compiler_options={"scalar_type": dtype})  # type: ignore
kernelf0 = getattr(ufcxf0.form_integrals[0], f"tabulate_tensor_{np.dtype(dtype).name}")  # type: ignore

ffi = cffi.FFI()

In [ ]:
dofmap, dummy_dof_index = hs.build_global_dof_map()
dummy_dof_index += 1 
num_control_points_with_dummy = dummy_dof_index+1

num_cells = disconnected_mesh.topology.index_map(disconnected_mesh.topology.dim).size_global

# which BSpline functions are active on each cell, padded with the dummy index.
padded_cells_to_dofs = np.full((num_cells, N_max), dummy_dof_index, dtype=np.int32)

# Find the maximum local index for each level to size arrays
max_local_indices = [0] * hs.nlevels
for l, local_idx in dofmap.keys():
    if local_idx > max_local_indices[l]:
        max_local_indices[l] = local_idx
    pass
pass

# convert the dictionary into a list of numpy arrays for faster lookup
dofmap_arrays = [np.full(size + 1, dummy_dof_index, dtype=np.int32) for size in max_local_indices]
for (l, local_idx), global_idx in dofmap.items():
    dofmap_arrays[l][local_idx] = global_idx


cell_index = 0
for l in range(hs.nlevels):
    active_cells_l = hs.hmesh.aelem_level[l]
    for cell in active_cells_l:
        
        # Get the local functions on the cell
        active_funcs_dict: dict[int, npt.NDArray[np.int_]] = hs.get_all_active_functions_on_cell(l, cell)
        
            
        #  Vectorized conversion from local to global indices
        mapped_arrays = [
            dofmap_arrays[ll][local_funcs] for ll, local_funcs in active_funcs_dict.items() if len(local_funcs) > 0
        ]
        
        #
        if mapped_arrays: # Check if there are actually active functions
            global_dofs = np.concatenate(mapped_arrays)
            n_dofs = len(global_dofs)
            padded_cells_to_dofs[cell_index, :n_dofs] = global_dofs
            
        cell_index += 1
new_pctd = padded_cells_to_dofs
# for i in range(len(new_pctd)):
#     row = new_pctd[i, :]
#     no_dummy = row[row<dummy_dof_index]
#     my_median = int(np.median(no_dummy))
#     new_pctd[i, :][new_pctd[i, :]==dummy_dof_index] = my_median


In [ ]:
def find_level_idx_from_midpoint(hs, midpoint, all_midpoints):
    truth_array = np.all(np.isclose(midpoint,all_midpoints), axis=-1)
    global_index = np.nonzero(truth_array)[0][0]
    level=0
    current_elts = hs.hmesh.aelem_level[level]
    #print(f"\nglobal_index = {global_index}")
    while(global_index>len(current_elts)-1):
        global_index-=len(current_elts)
        level+=1
        current_elts = hs.hmesh.aelem_level[level]
        #print(f"in for loop, global_index = {global_index}")
    #print(f"\n")
    return (level, hs.hmesh.aelem_level[level][global_index])

In [ ]:
M = hs._bezier_to_legendre(degree = p0)
S_indices = np.arange(p0+1, dtype=np.float64)
# scaling for unnormalised Legendre basis polynomials
S_inv = (1./np.sqrt(2.*S_indices+1.))*np.identity(p0+1, dtype=np.float64) # Scaling factor, since fenicsx uses orthonormal legendre polynomials
T = np.asfortranarray(np.kron(M, M).T @ np.kron(S_inv, S_inv), dtype=dtype)
local_dofs_size = T.shape[1]

operator_shape = (N_max, local_dofs_size)
# Create a custom space that holds the content of each matrix for the relevant cell.
# degree 0 because the value is constant over each cell
C_space = dolfinx.fem.functionspace(disconnected_mesh, ("DG", 0, operator_shape))
C_func = dolfinx.fem.Function(C_space, dtype=dtype)

num_cells_local = disconnected_mesh.topology.index_map(disconnected_mesh.topology.dim).size_local
indices = np.arange(num_cells_local, dtype=np.int32)
# To make sure that each matrix is assigned to the correct cell
midpoints: npt.NDArray[np.float_] = dolfinx.mesh.compute_midpoints(disconnected_mesh, disconnected_mesh.topology.dim, indices)

c_values = C_func.x.array.reshape((-1, N_max, T.shape[1]))
for local_idx, midpoint in enumerate(midpoints):
    #print(f"midpoint = {midpoint[:2]}")
    #level, idx = hs.hmesh.find_active_cell(midpoint[:hs.dim])
    level, idx = find_level_idx_from_midpoint(hs, midpoint=midpoint[:hs.dim], all_midpoints=physical_cells_midpoints)
    #print(f"level={level}, idx={idx}\n")

    mat = thb_operators[level, idx] @ hs.level_spaces[level].get_bezier_operator(idx)
    
    real_k, n_cols = mat.shape

    if real_k<N_max:
        padding_size = N_max - real_k
        to_pad = np.zeros((padding_size, mat.shape[1]), dtype=np.float64)
        mat_padded = np.vstack((mat, np.zeros((padding_size, mat.shape[1])) ))
        Ci = mat_padded
    else:
        Ci = mat
        #to_pad = np.zeros((0, mat.shape[1]))
    
    # is a view of C_func.x.array, therefore we modify the content of C_func.x.array
    # No new array is created, the matrix->cell mapping is done here.
    c_values[local_idx, :, :] = Ci@T# np.vstack((mat@T, to_pad))
C_func.x.scatter_forward()

num_control_points = np.max(new_pctd)+1
my_index_map = dolfinx.common.IndexMap(comm=disconnected_mesh.comm, 
                                       local_size=num_control_points)

dummy_element = basix.ufl.element(
    family="DG", 
    cell="quadrilateral", 
    degree=0, 
    shape=(N_max,)
)
dummy_space = dolfinx.fem.functionspace(mesh=disconnected_mesh, 
                                        element=dummy_element)
element_layout = dummy_space.dofmap.dof_layout
cpp_element = dummy_space.element._cpp_object

dof_indices = padded_cells_to_dofs.ravel().astype(np.int32)
offsets = (np.arange(num_cells + 1, dtype=np.int32) * N_max).astype(np.int32)

adj = dolfinx.cpp.graph.AdjacencyList_int32(data=dof_indices, 
                                            offsets=offsets)

doflinx_dofmap = dolfinx.cpp.fem.DofMap(
    element_dof_layout=element_layout, 
    index_map=my_index_map,  
    index_map_bs=1, 
    dofmap=adj, 
    bs=1
)

V_spline_cpp = dolfinx.cpp.fem.FunctionSpace_float64(
    mesh=disconnected_mesh._cpp_object, 
    element=cpp_element, 
    dofmap=doflinx_dofmap
)

V_spline = dolfinx.fem.FunctionSpace(
    mesh=disconnected_mesh, 
    element=dummy_element, 
    cppV=V_spline_cpp
)

In [ ]:
PADDED_DOFS = N_max
LOCAL_DOFS = local_dofs_size

@numba.cfunc(ufcx_signature(dtype, rtype), nopython=True)  # type: ignore
def tabulate_A(A_, w_, c_, coords_, entity_local_index, permutation=ffi.NULL, custom_data=None):

    # Prepare target condensed local element tensor
    # arguments: ptr, shape, dtype
    # returns a view over the original array A_
    # This has to be larger since we are working with padded arrays. Irrelevant dofs are mapped to a dummy location
    A = numba.carray(A_, (PADDED_DOFS, PADDED_DOFS), dtype=dtype)

    # Get the operator (TRUNC @ C_{B->BS} @ (C_{L->B}.T) @ S^{-1}) for this cell
    # TRUNC has shape (PADDED_DOFS, LOCAL_DOFS) and all other matrices have shape (LOCAL_DOFS, LOCAL_DOFS)
    G = numba.carray(w_, (PADDED_DOFS, LOCAL_DOFS), dtype=dtype)
    
    # Tabulate all sub blocks locally
    # This matrix is formed via the Legendre elements on a quadrilateral of degree p0,
    # therefore this has the shape (LOCAL_DOFS, LOCAL_DOFS)
    A0 = np.zeros((LOCAL_DOFS, LOCAL_DOFS), dtype=dtype)
    kernela0(
        ffi.from_buffer(A0),
        w_,# weights. Ignored for the L2 approximation because 
        # a0 does not depend on f and no value is looked up at the quadrature points
        # This is not very robust, one must be careful when calling this function
        c_,
        coords_,
        entity_local_index,
        permutation,
        empty_void_pointer(),
    )
    
    A[:, :] = G@A0@(G.T) 

@numba.cfunc(ufcx_signature(dtype, rtype), nopython=True)
def tabulate_b(b_, w_, c_, coords_, entity_local_index, permutation=ffi.NULL, custom_data=None):
    
    # Prepare target condensed local element tensor
    # arguments: ptr, shape, dtype
    # evaluation of f using padded THB-Splines
    b = numba.carray(b_, (PADDED_DOFS,), dtype=dtype)

    total_coeffs = LOCAL_DOFS*(PADDED_DOFS+1)
    # Evaluation of f at relevant points + values of change of basis matrix G
    w_flat = numba.carray(w_, (total_coeffs), dtype=dtype)

    # The following line breaks in case we do not pass [f._cpp_object, C_func._cpp_object] in this
    # exact order or if f does not have the correct amount of dofs.
    G = w_flat[LOCAL_DOFS:].reshape((PADDED_DOFS, LOCAL_DOFS))

    b0 = np.zeros((LOCAL_DOFS,), dtype=dtype)
    kernelf0(ffi.from_buffer(b0),
        w_, # This line only works because we specified that b0 is of size LOCAL_DOFS,
        # which happens to also be the size of f._cpp_object
        c_,
        coords_,
        entity_local_index,
        permutation,
        empty_void_pointer(),
    )
    # G@b0
    b[:] = G@b0

In [ ]:
facet_dim = msh.topology.dim-1
# fenicsx numbers cell facets (=edges) internally, as
# 0 for bottom, 1 for left, 2 for right and 3 for top 
# since all cells are disconnected, there should be in total 
# 4*n_cells boundary facets.
boundary_facets = dolfinx.mesh.locate_entities_boundary(msh, facet_dim, outer_boundary)
msh.topology.create_connectivity(facet_dim, msh.topology.dim)
msh.topology.create_connectivity(msh.topology.dim, facet_dim)

# dictionary of facet->cell
f_to_c = msh.topology.connectivity(facet_dim, msh.topology.dim)
# dictionary of cell->array[facets]
c_to_f = msh.topology.connectivity(msh.topology.dim, facet_dim)

boundary_entities = []
# Loop over all edges that belong to our exterior
for ff in boundary_facets:
    # returns the cells linked to this edge
    cells = f_to_c.links(ff)
    # Exterior facets only have 1 attached cell, 
    # hence get the first one 
    c = cells[0] 
    
    # Find the local index (e.g., 0, 1, 2, or 3 for quads) of facet f within cell c
    local_f = np.where(c_to_f.links(c) == ff)[0][0]
    #print(f"f={f}, cell={c}, local_f = {local_f}")
    
    boundary_entities.extend([c, local_f])

# FEniCSx custom form arrays must be typed as int32
boundary_entities = np.array(boundary_entities, dtype=np.int32)

In [ ]:
formtype = dolfinx.fem.form_cpp_class(dtype)  # type: ignore
# Gets the number of cells for which each individual core is responsible for.
cells = np.arange(msh.topology.index_map(msh.topology.dim).size_local, dtype=np.int32)

# The 4th argument np.array([...], dtype=np.int8) is the 
# active coefficients array. It lists which indices from the 
# coefficients list should be packed into the w_ pointer that the kernel receives.
integrals = {dolfinx.fem.IntegralType.cell: [
    (0, tabulate_A.address, cells, np.array([0], dtype=np.int8))]}

a_cond = dolfinx.fem.Form( # We are not forming anything yet, this is a recipe
    formtype( # selectes the correct floating-point precision
        spaces=[V_spline._cpp_object, 
                V_spline._cpp_object]
            , # trial and test spaces, determines the size of A_
        integrals=integrals, #this is a dictionary, and we are passing the adress of tabulate_A() here
        coefficients=[C_func._cpp_object
                    ], # weights w_, holds C@T
              constants=[],
              need_permutation_data=False,
              entity_maps=[], 
              mesh=msh._cpp_object)
)

integrals_rhs = {dolfinx.fem.IntegralType.cell: [(0, tabulate_b.address, cells, np.array([0,1], dtype=np.int8))]}
l_cond = dolfinx.fem.Form(
    formtype(
        spaces=[V_spline._cpp_object], # test space, determines the size of b_
        integrals=integrals_rhs, #give the adress of tabulate_b
        coefficients=[f._cpp_object, C_func._cpp_object], # holds the evaluations of f at the correct points, as well as C@T
        constants=[], need_permutation_data=False, entity_maps=[], mesh=msh._cpp_object
    )
)

In [ ]:
def get_spline_indices_on_inner_corner_turn(hs, dofmap):
    dirichlet_indices = {}
    for level in range(hs.nlevels):
        hs.level_spaces[level].construct_basis()
        basis = hs.level_spaces[level].basis
        current_dirichlet_indices = np.isclose(basis[:, 0, :-1], np.zeros((hs.degrees[0]+1)))
        current_dirichlet_indices = np.all(current_dirichlet_indices, axis=-1)
        current_dirichlet_indices = np.nonzero(current_dirichlet_indices)[0]
        dirichlet_indices[level] = np.intersect1d(hs.truly_active[level],
                                                  current_dirichlet_indices,
                                                  assume_unique=True)
    pass

    forbidden_indices = np.array([dofmap[level,idx] for level in dirichlet_indices 
              for idx in dirichlet_indices[level]],
                        dtype=np.int32)
    return forbidden_indices

forbidden_indices = get_spline_indices_on_inner_corner_turn(hs, dofmap)

In [ ]:
from dolfinx.fem.petsc import assemble_matrix, assemble_vector
from petsc4py import PETSc
# a_form = dolfinx.fem.form(a0)
A = assemble_matrix(a_cond, bcs=[])
A.assemble()
one_active=False
two_active= False
for level in range(hs.nlevels):
    if level in hs.truly_active and hs.truly_active[level].size>0:
        if one_active:
            two_active=True
        one_active=True

A_mat = A
if two_active:
    A_mat.setValue(dummy_dof_index, dummy_dof_index, 1., addv=PETSc.InsertMode.INSERT_VALUES)
    A_mat.assemble()
    A_mat.assemblyBegin()
    A_mat.assemblyEnd()
b = assemble_vector(l_cond)
if two_active:
    b[dummy_dof_index]=0.
    b.assemblyBegin()
    b.assemblyEnd()

ksp = PETSc.KSP().create(A.comm)
ksp.setOperators(A)
ksp.setType(PETSc.KSP.Type.PREONLY)
ksp.getPC().setType(PETSc.PC.Type.LU)
ksp.getPC().setFactorSolverType("mumps")
u_sol = dolfinx.fem.Function(V_spline)
ksp.solve(b, u_sol.x.petsc_vec)
u_sol.x.scatter_forward()
x_vec=u_sol.x.array
print(f"Solve complete. Reason: {ksp.getConvergedReason()}, Iterations: {ksp.getIterationNumber()}")

In [ ]:
u_dg = dolfinx.fem.Function(V)

# Map the global B-spline coefficients back to local Legendre coefficients
for local_idx in range(num_cells_local):
    # Get global B-spline dof indices for this cell
    spline_dofs = padded_cells_to_dofs[local_idx]
    
    # Extract the B-spline coefficients for this cell
    u_spline_local = x_vec[spline_dofs]
    
    # Get the local transformation matrix G for this cell
    G = c_values[local_idx, :, :]
    
    # Transform B-spline to DG: mathematically, the kernel does A = G @ A0 @ G.T
    # This implies the coefficient mapping is u_dg = G.T @ u_spline
    u_dg_local = G.T @ u_spline_local
    
    # Assign to the standard DG function
    dg_dofs = V.dofmap.cell_dofs(local_idx)
    u_dg.x.array[dg_dofs] = u_dg_local

u_dg.x.scatter_forward()


# Compute exact L2 error using FEniCSx standard UFL
error_form = dolfinx.fem.form(ufl.inner(f - u_dg, f - u_dg) * dx_custom)
error_sq = dolfinx.fem.assemble_scalar(error_form)
exact_l2_error = np.sqrt(disconnected_mesh.comm.allreduce(error_sq, op=MPI.SUM))

print(f"Exact L2 Error (via DG projection): {exact_l2_error:.2e}")
print(f"Relative error = {exact_l2_error/f_sq_integral:.2e}")

ksp.destroy()
A.destroy()
b.destroy()

In [ ]:
# Create a DG0 space (one value per cell)
V_error = dolfinx.fem.functionspace(disconnected_mesh, ("DG", 0))
v = ufl.TestFunction(V_error)
hQ = ufl.CellDiameter(disconnected_mesh)
volume_form = dolfinx.fem.form(1.0*v*dx_custom)
cell_volumes = dolfinx.fem.assemble_vector(volume_form).array
#print(f"cell_volumes = {cell_volumes[:10]}")
# Define the local L2 error form: integral of (f - u_dg)^2 per cell
# Note: We multiply by the test function 'v' to pick out each cell's contribution
local_error_form = dolfinx.fem.form(ufl.inner(f - u_dg, f - u_dg) *v * dx_custom)

# Assemble the vector (this gives us the squared error per cell)
local_error_vector = dolfinx.fem.assemble_vector(local_error_form)
rms_cell_errors = np.sqrt(local_error_vector.array /cell_volumes)
local_error_vector.scatter_forward()


# Take the square root to get the L2 norm per cell
# (Using .array for NumPy manipulation)
cell_errors = np.sqrt(local_error_vector.array)
squared_errors = cell_errors**2
total_squared_error = np.sum(squared_errors)
descending_indices = np.flip(np.argsort(squared_errors))
sorted_squared_errors = squared_errors[descending_indices]
cumulative_errors = np.cumsum(sorted_squared_errors)
theta = 0.2
threshold_value = theta*total_squared_error
num_cells_to_mark = max(2, np.searchsorted(cumulative_errors, threshold_value)+1)
top_error_indices = descending_indices[:num_cells_to_mark]
print(f"Total cells marked via Dörfler (theta={theta}): {num_cells_to_mark} out of {len(cell_errors)}")
print(f"Indices to refine: {top_error_indices[:10]}")

# # Calculate the threshold for the top 20%
# proportion = 0.17
# num_cells = len(cell_errors)
# top_percent_count = int(max(4, num_cells *proportion))

# # Get the indices of the cells sorted by error (ascending)
# sorted_indices = np.argsort(cell_errors)

# # Extract the indices of the top 25%
# top_error_indices = sorted_indices[-top_percent_count:]

# print(f"Top {proportion*100}% error threshold: {cell_errors[sorted_indices[-top_percent_count]]:.4e}")
# print(f"Indices of top {proportion*100}% error cells: {top_error_indices[:10]}")
# sorted_rms = np.argsort(rms_cell_errors)
# top_error_rms = sorted_rms[-top_percent_count:]
# print(f"rms = {top_error_rms}")

my_arr = []
for value in hs.hmesh.aelem_level.values():
    my_arr.extend(value)
err_cells = {}
sorted_top_error_indices = np.sort(top_error_indices)
#print(sorted_top_error_indices)
level=0
#go_back = 0
current_length = len(hs.hmesh.aelem_level[0])
for i in sorted_top_error_indices:
    # print(f"i={i}, current_length={current_length}")
    while i > current_length-1:
        level+=1
        current_length+=len(hs.hmesh.aelem_level[level])
    # local_i = i-go_back
    # if i>len(hs.hmesh.aelem_level[level]):
    #     go_back += len(hs.hmesh.aelem_level[level])
    #     level+=1
    #     local_i=i-go_back
    
    if level not in err_cells:
        err_cells[level]=[]
    #print(f"level={level}, i={i}, go_back={go_back}, local_i={local_i}")
    
    err_cells[level].append(my_arr[i])


# my_arr_rms = []
# for value in hs.hmesh.aelem_level.values():
#     my_arr_rms.extend(value)
# err_cells_rms = {}
# sorted_top_error_indices_rms = np.sort(top_error_rms)
# #print(sorted_top_error_indices)
# level=0
# #go_back = 0
# current_length = len(hs.hmesh.aelem_level[0])
# for i in sorted_top_error_indices_rms:
#     # print(f"i={i}, current_length={current_length}")
#     while i > current_length-1:
#         level+=1
#         current_length+=len(hs.hmesh.aelem_level[level])
#     # local_i = i-go_back
#     # if i>len(hs.hmesh.aelem_level[level]):
#     #     go_back += len(hs.hmesh.aelem_level[level])
#     #     level+=1
#     #     local_i=i-go_back
    
#     if level not in err_cells_rms:
#         err_cells_rms[level]=[]
#     #print(f"level={level}, i={i}, go_back={go_back}, local_i={local_i}")
    
#     err_cells_rms[level].append(my_arr_rms[i])

# print(f"err_cells_rms = {err_cells_rms}")

In [ ]:
import dolfinx.plot
import pyvista

# 1. Create a "Nodal" DG space of the same degree for plotting
# By default, DG with no variant specified uses Lagrange (nodal)
v_plot_elt = basix.ufl.element(
    "DG", 
    "quadrilateral", 
    degree=p0+2
)
V_plot = dolfinx.fem.functionspace(disconnected_mesh, v_plot_elt)

# 2. Interpolate your computed solution (u_dg) into the nodal space
#u_plot = dolfinx.fem.Function(V_plot)
error_ufl = ufl.ln(ufl.sqrt((u_dg-f)**2)+1e-13)
#error_ufl = u_dg
error_expr = dolfinx.fem.Expression(error_ufl, V_plot.element.interpolation_points)
u_error = dolfinx.fem.Function(V_plot)
u_error.interpolate(error_expr)

# 3. Now use V_plot for the VTK mesh generation
topology, cell_types, geometry = dolfinx.plot.vtk_mesh(V_plot)
grid = pyvista.UnstructuredGrid(topology, cell_types, geometry)

# 4. Attach the interpolated values
grid.point_data["u"] = u_error.x.array.real
grid.set_active_scalars("u")

# 5. Plotting (with a 'shrink' to see your disconnected mesh boundaries!)
plotter = pyvista.Plotter()
grid_shrink = grid.shrink(0.95) # This makes the "disconnected" nature visible
plotter.add_mesh(grid_shrink, show_edges=False, cmap="turbo")
plotter.view_xy()
plotter.show(jupyter_backend="static")
plotter.show()

# from pyvista.trame.jupyter import launch_server
# pyvista.set_jupyter_backend('client')
# warped_grid = grid.warp_by_scalar("u", factor=0.5) 

# # If you still want to see the gaps between cells:
# grid_shrink = warped_grid.shrink(0.95)

# plotter.add_mesh(grid_shrink, show_edges=False, cmap="viridis", lighting=True)

# # Set a nice 3D camera angle instead of view_xy()
# plotter.camera_position = 'iso' 
# await launch_server().ready
# plotter.show()